# Regressão linear simples

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_07/01_regressao_linear_simples.ipynb)

A regressão linear simples resume como uma resposta numérica varia, em média,
com uma variável explicativa. O diagrama recorda que a reta não elimina a distância
entre cada caso e a previsão.

![Pontos observados, reta estimada e distâncias verticais chamadas resíduos.](imagens/01_reta_residuos.svg)

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_07'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

## 1. Da pergunta às variáveis

Pergunta: **como a quantidade de equipamentos culturais por 100 mil habitantes
varia com a renda per capita no conjunto fictício?**

Definimos $Y$ como `equipamentos_culturais_100mil` e $X$ como `renda_pc`.
A unidade de análise é o município fictício. Primeiro inspecionamos forma, escala e
casos; só então resumimos a relação por uma reta.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

dados = pd.read_csv("dados/municipios_culturais.csv")
plt.scatter(dados["renda_pc"], dados["equipamentos_culturais_100mil"], alpha=.65, color="#426b78")
plt.xlabel("Renda per capita simulada")
plt.ylabel("Equipamentos por 100 mil")
plt.title("Renda e equipamentos — dados inteiramente fictícios")
plt.tight_layout(); plt.show()

## 2. A equação como argumento resumido

$$Y_i = \beta_0 + \beta_1 X_i + \varepsilon_i$$

O intercepto $\beta_0$ é o valor previsto quando $X=0$; pode não ter sentido
substantivo se zero estiver fora dos dados. A inclinação $\beta_1$ é a mudança média
prevista em $Y$ para uma unidade adicional de $X$. O resíduo $\varepsilon_i$ reúne a
parcela que esta reta não descreve.

A célula seguinte reúne os cálculos matriciais numa função reutilizável. Você não
precisa memorizar `lstsq`: acompanhe suas entradas (`X`, `y` e nomes) e as saídas
interpretáveis (coeficientes, previsões, resíduos, $R^2$, RMSE e MAE).

In [ ]:
def ajustar_ols(X, y, nomes):
    X_com_intercepto = np.column_stack([np.ones(len(X)), np.asarray(X, dtype=float)])
    coeficientes = np.linalg.lstsq(X_com_intercepto, np.asarray(y, dtype=float), rcond=None)[0]
    previstos = X_com_intercepto @ coeficientes
    residuos = np.asarray(y, dtype=float) - previstos
    r2 = 1 - np.sum(residuos ** 2) / np.sum((y - np.mean(y)) ** 2)
    return {
        "X": X_com_intercepto, "coeficientes": coeficientes,
        "previstos": previstos, "residuos": residuos,
        "r2": r2, "rmse": np.sqrt(np.mean(residuos ** 2)),
        "mae": np.mean(np.abs(residuos)), "nomes": ["intercepto", *nomes],
    }

def tabela_coeficientes(modelo):
    return pd.DataFrame({"termo": modelo["nomes"], "coeficiente": modelo["coeficientes"]})

In [ ]:
simples = ajustar_ols(
    dados[["renda_pc"]], dados["equipamentos_culturais_100mil"], ["renda_pc"]
)
display(tabela_coeficientes(simples).round(4))
print(f"R²: {simples['r2']:.3f} | RMSE: {simples['rmse']:.3f}")

ordem = np.argsort(dados["renda_pc"].to_numpy())
plt.scatter(dados["renda_pc"], dados["equipamentos_culturais_100mil"], alpha=.55)
plt.plot(dados["renda_pc"].to_numpy()[ordem], simples["previstos"][ordem], color="#a44a3f", linewidth=2)
plt.xlabel("Renda per capita simulada"); plt.ylabel("Equipamentos por 100 mil")
plt.title("Reta estimada sobre os casos observados")
plt.tight_layout(); plt.show()

## 3. Interpretar em escala compreensível

Como uma unidade monetária é pequena, multiplicamos a inclinação por 500. Essa
reexpressão muda a leitura, não o modelo. $R^2$ indica a proporção da variação de
$Y$ resumida pela reta neste conjunto; RMSE expressa o erro típico na unidade de $Y$.
Nenhuma medida demonstra causalidade ou qualidade da fonte. A última saída deste
notebook ordena resíduos absolutos para que a atividade consiga retornar aos casos,
em vez de interpretar apenas coeficientes agregados.

In [ ]:
inclinacao_500 = simples["coeficientes"][1] * 500
print(f"Associação estimada para +500 unidades de renda: {inclinacao_500:.2f} equipamento(s) por 100 mil.")
print(f"R²: {simples['r2']:.3f} | RMSE: {simples['rmse']:.3f} | MAE: {simples['mae']:.3f}")

casos_residuos = dados[["id_municipio", "municipio", "renda_pc", "equipamentos_culturais_100mil"]].copy()
casos_residuos["previsto"] = simples["previstos"]
casos_residuos["residuo"] = simples["residuos"]
casos_residuos["residuo_absoluto"] = np.abs(simples["residuos"])
display(casos_residuos.sort_values("residuo_absoluto", ascending=False).head(6).round(3))

### U07-A02 — Atividade — ler uma regressão simples

1. Declare unidade, resposta e explicativa.
2. Interprete a inclinação para 500 unidades de renda e o $R^2$.
3. Inspecione dois casos distantes da reta.
4. Escreva uma conclusão de até 90 palavras, incluindo um limite causal.

**Dinâmica (25 minutos):** trabalho individual por 15 minutos; depois, troque a
conclusão com uma colega e circule palavras fortes demais, como “causa” ou “prova”.

**Resposta:** escreva aqui.

## Passagem

A reta simples mistura renda com outras diferenças entre municípios. O próximo
notebook inclui essas características e ensina a interpretar a comparação
condicionada que daí resulta.

**Leitura:** Gelman, Hill e Vehtari (2020) e Fox (2016), listados em
`referencias.md`.